In [1]:
# Day 11: Recommendation Engine and Anomaly Detection
# Step 1: Import libraries and verify required datasets

from pathlib import Path
import pandas as pd
import numpy as np

# Project directory
PROJECT_ROOT = Path(r"D:\Nexzibyte Solution\nexzibyte-predictive-bi")

# Required datasets
files = {
    "customer_360": PROJECT_ROOT / "data" / "processed" / "Day10_Customer_360.csv",
    "customer_segments": PROJECT_ROOT / "data" / "processed" / "Day10_Customer_Segments.csv",
    "expansion_candidates": PROJECT_ROOT / "data" / "processed" / "Day10_Expansion_Candidates.csv",
    "quotations": PROJECT_ROOT / "data" / "synthetic" / "Day7_Synthetic_Quotation_Dataset.csv",
    "project_risk": PROJECT_ROOT / "data" / "synthetic" / "Day9_Synthetic_Project_Risk_Dataset.csv",
    "monthly_revenue": PROJECT_ROOT / "data" / "synthetic" / "Day8_Synthetic_Monthly_Revenue.csv",
}

# Check that the files exist
for name, path in files.items():
    status = "FOUND" if path.exists() else "MISSING"
    print(f"{name}: {status} | {path.name}")

print("\nProject root:", PROJECT_ROOT)
print("Project root exists:", PROJECT_ROOT.exists())

customer_360: FOUND | Day10_Customer_360.csv
customer_segments: FOUND | Day10_Customer_Segments.csv
expansion_candidates: FOUND | Day10_Expansion_Candidates.csv
quotations: FOUND | Day7_Synthetic_Quotation_Dataset.csv
project_risk: FOUND | Day9_Synthetic_Project_Risk_Dataset.csv
monthly_revenue: FOUND | Day8_Synthetic_Monthly_Revenue.csv

Project root: D:\Nexzibyte Solution\nexzibyte-predictive-bi
Project root exists: True


In [2]:
# Day 11: Step 2 - Load and inspect datasets

# Load all six CSV files
customer_360 = pd.read_csv(files["customer_360"])
customer_segments = pd.read_csv(files["customer_segments"])
expansion_candidates = pd.read_csv(files["expansion_candidates"])
quotations = pd.read_csv(files["quotations"])
project_risk = pd.read_csv(files["project_risk"])
monthly_revenue = pd.read_csv(files["monthly_revenue"])

# Display dataset dimensions and column names
datasets = {
    "Customer 360": customer_360,
    "Customer Segments": customer_segments,
    "Expansion Candidates": expansion_candidates,
    "Quotation Dataset": quotations,
    "Project Risk Dataset": project_risk,
    "Monthly Revenue Dataset": monthly_revenue,
}

for name, df in datasets.items():
    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)
    print("Rows:", df.shape[0])
    print("Columns:", df.shape[1])
    print("Column names:")
    print(df.columns.tolist())
    print("Missing values:", int(df.isnull().sum().sum()))


Customer 360
Rows: 300
Columns: 14
Column names:
['Company_ID', 'Quote_Count', 'First_Quote_Date', 'Last_Quote_Date', 'Total_Quoted_Value', 'Average_Quoted_Value', 'Won_Quote_Count', 'Lost_Quote_Count', 'Pending_Quote_Count', 'Recency_Days', 'Inactive_180D_Proxy', 'Assessment_Date', 'Synthetic_Data_Flag', 'Data_Use']
Missing values: 0

Customer Segments
Rows: 300
Columns: 21
Column names:
['Company_ID', 'Quote_Count', 'First_Quote_Date', 'Last_Quote_Date', 'Total_Quoted_Value', 'Average_Quoted_Value', 'Won_Quote_Count', 'Lost_Quote_Count', 'Pending_Quote_Count', 'Recency_Days', 'Inactive_180D_Proxy', 'Assessment_Date', 'Synthetic_Data_Flag', 'Data_Use', 'High_Quote_Value_Flag', 'Recent_Activity_Flag', 'Won_Quote_Flag', 'Customer_Segment', 'Segment_Method', 'Expansion_Candidate_Flag', 'Expansion_Candidate_Reason']
Missing values: 0

Expansion Candidates
Rows: 2
Columns: 21
Column names:
['Company_ID', 'Quote_Count', 'First_Quote_Date', 'Last_Quote_Date', 'Total_Quoted_Value', 'Average_

In [3]:
# Day 11: Step 3 - Inspect quotation data quality

print("QUOTATION DATA TYPES")
print("-" * 60)
print(quotations.dtypes)

print("\nMISSING VALUES BY COLUMN")
print("-" * 60)

missing_summary = pd.DataFrame({
    "Missing_Count": quotations.isnull().sum(),
    "Missing_Percentage": (
        quotations.isnull().mean() * 100
    ).round(2)
})

missing_summary = missing_summary.sort_values(
    by="Missing_Count",
    ascending=False
)

print(missing_summary)

print("\nQUOTATION STATUS COUNTS")
print("-" * 60)
print(
    quotations["Quote_Status"]
    .value_counts(dropna=False)
)

print("\nSYNTHETIC DATA LABELS")
print("-" * 60)
print(
    quotations[
        ["Synthetic_Data_Flag", "Data_Use"]
    ].drop_duplicates()
)

QUOTATION DATA TYPES
------------------------------------------------------------
Quote_ID                object
Company_ID              object
Quote_Date              object
Quote_Value              int64
Discount_Pct           float64
Revision_Count           int64
Engagement_Count         int64
Quote_Status            object
Accepted_Date           object
Loss_Reason             object
Synthetic_Data_Flag       bool
Data_Use                object
dtype: object

MISSING VALUES BY COLUMN
------------------------------------------------------------
                     Missing_Count  Missing_Percentage
Accepted_Date                  182               60.67
Loss_Reason                    163               54.33
Quote_ID                         0                0.00
Company_ID                       0                0.00
Quote_Value                      0                0.00
Quote_Date                       0                0.00
Discount_Pct                     0                0.00
Revis

In [4]:
# Day 11: Step 4 - Validate quotation missing values by status

status_missing_check = (
    quotations.groupby("Quote_Status")
    .agg(
        Total_Quotes=("Quote_ID", "count"),
        Missing_Accepted_Date=("Accepted_Date", lambda s: s.isna().sum()),
        Missing_Loss_Reason=("Loss_Reason", lambda s: s.isna().sum()),
        Missing_Accepted_Date_Pct=(
            "Accepted_Date",
            lambda s: round(s.isna().mean() * 100, 2)
        ),
        Missing_Loss_Reason_Pct=(
            "Loss_Reason",
            lambda s: round(s.isna().mean() * 100, 2)
        )
    )
    .reset_index()
)

print(status_missing_check.to_string(index=False))

Quote_Status  Total_Quotes  Missing_Accepted_Date  Missing_Loss_Reason  Missing_Accepted_Date_Pct  Missing_Loss_Reason_Pct
        Lost           137                    137                    0                      100.0                      0.0
     Pending            45                     45                   45                      100.0                    100.0
         Won           118                      0                  118                        0.0                    100.0


In [5]:
# Day 11: Step 5 - Customer Recommendation Engine

# Work on a copy so existing datasets remain unchanged
recommendations = customer_segments.copy()

# Define recommendation rules
def recommend_next_action(row):
    # Rule 1: Customers flagged as expansion candidates
    if row["Expansion_Candidate_Flag"]:
        return pd.Series({
            "Recommended_Action": "Explore expansion opportunity",
            "Recommendation_Reason": row["Expansion_Candidate_Reason"],
            "Priority": "High"
        })

    # Rule 2: Pending quotations need follow-up
    if row["Pending_Quote_Count"] > 0:
        return pd.Series({
            "Recommended_Action": "Follow up on pending quotation",
            "Recommendation_Reason": "One or more quotations remain pending",
            "Priority": "High"
        })

    # Rule 3: Inactive customers need re-engagement
    if row["Inactive_180D_Proxy"] == 1:
        return pd.Series({
            "Recommended_Action": "Review customer re-engagement",
            "Recommendation_Reason": "Customer is flagged by the 180-day inactivity proxy",
            "Priority": "Medium"
        })

    # Rule 4: Customers with a won quotation
    if row["Won_Quote_Count"] > 0:
        return pd.Series({
            "Recommended_Action": "Explore relevant follow-on services",
            "Recommendation_Reason": "Customer has at least one won quotation",
            "Priority": "Medium"
        })

    # Rule 5: Default review action
    return pd.Series({
        "Recommended_Action": "Review customer needs and quotation history",
        "Recommendation_Reason": "No higher-priority rule matched",
        "Priority": "Low"
    })


# Apply rules to every customer
recommendation_details = recommendations.apply(
    recommend_next_action,
    axis=1
)

recommendations = pd.concat(
    [recommendations, recommendation_details],
    axis=1
)

# Add a clear synthetic-data disclaimer
recommendations["Recommendation_Data_Status"] = (
    "Illustrative recommendation based on synthetic data"
)

# Display a preview
print("Recommendation table shape:", recommendations.shape)

print("\nRecommended action counts:")
print(recommendations["Recommended_Action"].value_counts())

print("\nPriority counts:")
print(recommendations["Priority"].value_counts())

print("\nSample recommendations:")
display(
    recommendations[
        [
            "Company_ID",
            "Customer_Segment",
            "Recommended_Action",
            "Recommendation_Reason",
            "Priority",
            "Recommendation_Data_Status"
        ]
    ].head(10)
)

Recommendation table shape: (300, 25)

Recommended action counts:
Recommended_Action
Review customer re-engagement                  117
Review customer needs and quotation history     75
Explore relevant follow-on services             61
Follow up on pending quotation                  45
Explore expansion opportunity                    2
Name: count, dtype: int64

Priority counts:
Priority
Medium    178
Low        75
High       47
Name: count, dtype: int64

Sample recommendations:


,Company_ID,Customer_Segment,Recommended_Action,Recommendation_Reason,Priority,Recommendation_Data_Status
0,SYN-C-0161,Inactive Quote,Review customer re-engagement,Customer is flagged by the 180-day inactivity ...,Medium,Illustrative recommendation based on synthetic...
1,SYN-C-0145,Inactive Quote,Review customer re-engagement,Customer is flagged by the 180-day inactivity ...,Medium,Illustrative recommendation based on synthetic...
2,SYN-C-0265,High Quote Value,Follow up on pending quotation,One or more quotations remain pending,High,Illustrative recommendation based on synthetic...
3,SYN-C-0141,Inactive Quote,Review customer re-engagement,Customer is flagged by the 180-day inactivity ...,Medium,Illustrative recommendation based on synthetic...
4,SYN-C-0012,Inactive Quote,Review customer re-engagement,Customer is flagged by the 180-day inactivity ...,Medium,Illustrative recommendation based on synthetic...
5,SYN-C-0046,Inactive Quote,Review customer re-engagement,Customer is flagged by the 180-day inactivity ...,Medium,Illustrative recommendation based on synthetic...
6,SYN-C-0208,High Quote Value,Review customer needs and quotation history,No higher-priority rule matched,Low,Illustrative recommendation based on synthetic...
7,SYN-C-0064,Inactive Quote,Review customer re-engagement,Customer is flagged by the 180-day inactivity ...,Medium,Illustrative recommendation based on synthetic...
8,SYN-C-0200,Inactive Quote,Review customer re-engagement,Customer is flagged by the 180-day inactivity ...,Medium,Illustrative recommendation based on synthetic...
9,SYN-C-0195,Inactive Quote,Review customer re-engagement,Customer is flagged by the 180-day inactivity ...,Medium,Illustrative recommendation based on synthetic...


In [6]:
# Day 11: Step 6 - Validate customer recommendations

validation_checks = {
    "Expected customer count is 300": len(recommendations) == 300,

    "Company_ID has no missing values":
        recommendations["Company_ID"].notna().all(),

    "Company_ID values are unique":
        recommendations["Company_ID"].is_unique,

    "Recommended action has no missing values":
        recommendations["Recommended_Action"].notna().all(),

    "Recommendation reason has no missing values":
        recommendations["Recommendation_Reason"].notna().all(),

    "Priority has no missing values":
        recommendations["Priority"].notna().all(),

    "Priority values are valid":
        recommendations["Priority"].isin(
            ["High", "Medium", "Low"]
        ).all(),

    "All recommendations have synthetic-data disclaimer":
        recommendations["Recommendation_Data_Status"].eq(
            "Illustrative recommendation based on synthetic data"
        ).all(),
}

print("CUSTOMER RECOMMENDATION VALIDATION")
print("=" * 60)

for check_name, passed in validation_checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {check_name}")

print("=" * 60)

if all(validation_checks.values()):
    print("RESULT: ALL VALIDATION CHECKS PASSED")
else:
    print("RESULT: VALIDATION FAILED - REVIEW THE CHECKS ABOVE")

CUSTOMER RECOMMENDATION VALIDATION
PASS: Expected customer count is 300
PASS: Company_ID has no missing values
PASS: Company_ID values are unique
PASS: Recommended action has no missing values
PASS: Recommendation reason has no missing values
PASS: Priority has no missing values
PASS: Priority values are valid
PASS: All recommendations have synthetic-data disclaimer
RESULT: ALL VALIDATION CHECKS PASSED


In [7]:
# Day 11: Step 7 - Save customer recommendation outputs

from pathlib import Path

# Output directory
output_dir = PROJECT_ROOT / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

# File 1: Complete recommendation table
recommendation_file = (
    output_dir / "Day11_Customer_Recommendations.csv"
)

recommendations.to_csv(
    recommendation_file,
    index=False
)

# File 2: Recommendation summary by action and priority
recommendation_summary = (
    recommendations
    .groupby(
        ["Recommended_Action", "Priority"],
        dropna=False
    )
    .size()
    .reset_index(name="Customer_Count")
)

recommendation_summary_file = (
    output_dir / "Day11_Recommendation_Summary.csv"
)

recommendation_summary.to_csv(
    recommendation_summary_file,
    index=False
)

# Confirm saved files
print("CUSTOMER RECOMMENDATION OUTPUTS")
print("=" * 60)

for path in [recommendation_file, recommendation_summary_file]:
    print(f"File: {path.name}")
    print(f"Exists: {path.exists()}")
    print(f"Size: {path.stat().st_size:,} bytes")
    print()

# Read back the saved files and verify row counts
saved_recommendations = pd.read_csv(recommendation_file)
saved_summary = pd.read_csv(recommendation_summary_file)

print("Saved recommendation rows:", len(saved_recommendations))
print("Saved summary rows:", len(saved_summary))

assert len(saved_recommendations) == 300
assert saved_recommendations["Company_ID"].is_unique

print("\nPASS: Recommendation outputs saved and verified.")


CUSTOMER RECOMMENDATION OUTPUTS
File: Day11_Customer_Recommendations.csv
Exists: True
Size: 122,239 bytes

File: Day11_Recommendation_Summary.csv
Exists: True
Size: 263 bytes

Saved recommendation rows: 300
Saved summary rows: 5

PASS: Recommendation outputs saved and verified.


In [8]:
# Day 11: Step 8.1 - Inspect quotation metrics

quotation_metrics = [
    "Quote_Value",
    "Discount_Pct",
    "Revision_Count",
    "Engagement_Count",
]

print("QUOTATION METRIC SUMMARY")
print("=" * 70)

display(
    quotations[quotation_metrics]
    .describe()
    .round(2)
)

print("\nQUOTATION METRICS BY STATUS")
print("=" * 70)

display(
    quotations.groupby("Quote_Status")[quotation_metrics]
    .agg(["count", "mean", "median", "min", "max"])
    .round(2)
)

print("\nSYNTHETIC DATA CHECK")
print("=" * 70)

print(
    "All records marked synthetic:",
    quotations["Synthetic_Data_Flag"].eq(True).all()
)

print(
    "Data-use labels:",
    quotations["Data_Use"].unique().tolist()
)

QUOTATION METRIC SUMMARY


,Quote_Value,Discount_Pct,Revision_Count,Engagement_Count
count,300.00,300.00,300.00,300.00
mean,257098.23,12.62,2.01,5.06
std,132107.48,7.50,1.36,3.20
min,18570.00,0.14,0.00,0.00
25%,149202.50,5.80,1.00,2.00
50%,254288.00,13.08,2.00,5.00
75%,366968.75,19.49,3.00,8.00
max,498112.00,24.98,4.00,10.00



QUOTATION METRICS BY STATUS


Quote_Value                                     Discount_Pct  \
                   count       mean    median    min     max        count   
Quote_Status                                                                
Lost                 137  268129.04  279327.0  26059  498112          137   
Pending               45  260354.60  256429.0  25481  494997           45   
Won                  118  243049.44  236812.5  18570  496443          118   

                                        Revision_Count                       \
               mean median   min    max          count  mean median min max   
Quote_Status                                                                  
Lost          11.19   9.94  0.90  24.67            137  2.21    2.0   0   4   
Pending       12.73  13.85  0.14  24.16             45  2.07    2.0   0   4   
Won           14.23  15.52  0.31  24.98            118  1.75    2.0   0   4   

             Engagement_Count                       
                        count  mean median min max  
Quote_Status                                        
Lost                      137  4.52    4.0   0  10  
Pending                    45  4.76    4.0   0  10  
Won                       118  5.81    6.0   0  10


SYNTHETIC DATA CHECK
All records marked synthetic: True
Data-use labels: ['Demonstration only; not real business data']


In [9]:
# Day 11: Step 8.2 - Detect quotation metric outliers using IQR

# Work on a copy; do not modify the original quotation dataset
quotation_anomalies = quotations.copy()

metrics = [
    "Quote_Value",
    "Discount_Pct",
    "Revision_Count",
    "Engagement_Count",
]

# Create anomaly flags and calculate IQR thresholds
iqr_thresholds = []

for metric in metrics:
    q1 = quotation_anomalies[metric].quantile(0.25)
    q3 = quotation_anomalies[metric].quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    flag_column = f"{metric}_IQR_Anomaly"

    quotation_anomalies[flag_column] = (
        (quotation_anomalies[metric] < lower_bound)
        | (quotation_anomalies[metric] > upper_bound)
    )

    iqr_thresholds.append({
        "Metric": metric,
        "Q1": round(q1, 2),
        "Q3": round(q3, 2),
        "IQR": round(iqr, 2),
        "Lower_Bound": round(lower_bound, 2),
        "Upper_Bound": round(upper_bound, 2),
        "Anomaly_Count": int(
            quotation_anomalies[flag_column].sum()
        ),
    })

# Display thresholds and counts
threshold_summary = pd.DataFrame(iqr_thresholds)

print("IQR THRESHOLDS AND ANOMALY COUNTS")
print("=" * 80)
display(threshold_summary)

# Count how many metrics are flagged for each quotation
anomaly_columns = [
    f"{metric}_IQR_Anomaly" for metric in metrics
]

quotation_anomalies["Anomaly_Metric_Count"] = (
    quotation_anomalies[anomaly_columns].sum(axis=1)
)

quotation_anomalies["Anomaly_Flag"] = (
    quotation_anomalies["Anomaly_Metric_Count"] > 0
)

print("\nOVERALL RESULTS")
print("=" * 80)
print("Total quotations:", len(quotation_anomalies))
print(
    "Quotations with at least one IQR flag:",
    int(quotation_anomalies["Anomaly_Flag"].sum())
)
print(
    "Quotations without IQR flags:",
    int((~quotation_anomalies["Anomaly_Flag"]).sum())
)

print("\nANOMALY COUNTS BY QUOTATION STATUS")
display(
    quotation_anomalies.groupby("Quote_Status")[
        ["Anomaly_Flag", "Anomaly_Metric_Count"]
    ].agg({
        "Anomaly_Flag": "sum",
        "Anomaly_Metric_Count": "sum",
    })
)

print("\nSAMPLE FLAGGED QUOTATIONS")
display(
    quotation_anomalies.loc[
        quotation_anomalies["Anomaly_Flag"],
        [
            "Quote_ID",
            "Quote_Status",
            "Quote_Value",
            "Discount_Pct",
            "Revision_Count",
            "Engagement_Count",
            "Anomaly_Metric_Count",
        ],
    ].head(10)
)

IQR THRESHOLDS AND ANOMALY COUNTS


,Metric,Q1,Q3,IQR,Lower_Bound,Upper_Bound,Anomaly_Count
0,Quote_Value,149202.5,366968.75,217766.25,-177446.88,693618.12,0
1,Discount_Pct,5.8,19.49,13.70,-14.75,40.04,0
2,Revision_Count,1.0,3.00,2.00,-2.00,6.00,0
3,Engagement_Count,2.0,8.00,6.00,-7.00,17.00,0



OVERALL RESULTS
Total quotations: 300
Quotations with at least one IQR flag: 0
Quotations without IQR flags: 300

ANOMALY COUNTS BY QUOTATION STATUS


,Anomaly_Flag,Anomaly_Metric_Count
Quote_Status,,
Lost,0,0
Pending,0,0
Won,0,0



SAMPLE FLAGGED QUOTATIONS


,Quote_ID,Quote_Status,Quote_Value,Discount_Pct,Revision_Count,Engagement_Count,Anomaly_Metric_Count


In [10]:
# Day 11: Step 9.1 - Business-rule quotation review flags

# Work with the anomaly dataset created in Step 8.2
quotation_review = quotation_anomalies.copy()

# Calculate a data-driven screening threshold for high quotation values
quote_value_p95 = quotation_review["Quote_Value"].quantile(0.95)

# Define transparent screening rules
quotation_review["High_Discount_Review"] = (
    quotation_review["Discount_Pct"] >= 20
)

quotation_review["High_Revisions_Review"] = (
    quotation_review["Revision_Count"] >= 4
)

quotation_review["Zero_Engagement_Review"] = (
    quotation_review["Engagement_Count"] == 0
)

quotation_review["High_Quote_Value_Review"] = (
    quotation_review["Quote_Value"] >= quote_value_p95
)

# Count business-rule flags for each quotation
review_columns = [
    "High_Discount_Review",
    "High_Revisions_Review",
    "Zero_Engagement_Review",
    "High_Quote_Value_Review",
]

quotation_review["Business_Rule_Flag_Count"] = (
    quotation_review[review_columns].sum(axis=1)
)

quotation_review["Business_Rule_Review_Flag"] = (
    quotation_review["Business_Rule_Flag_Count"] > 0
)

# Label the outputs appropriately
quotation_review["Review_Status"] = np.where(
    quotation_review["Business_Rule_Review_Flag"],
    "Review recommended",
    "No rule triggered"
)

quotation_review["Data_Status"] = (
    "Synthetic demonstration; flags require human review"
)

# Summarize each rule
rule_summary = pd.DataFrame({
    "Rule": review_columns,
    "Flagged_Quotes": [
        int(quotation_review[column].sum())
        for column in review_columns
    ],
})

print("BUSINESS-RULE REVIEW THRESHOLDS")
print("=" * 65)
print(f"95th percentile of quote value: {quote_value_p95:.2f}")
print("High discount threshold: 20%")
print("High revisions threshold: 4 or more")
print("Zero engagement threshold: 0")
print()

print("RULE SUMMARY")
display(rule_summary)

print("\nOVERALL REVIEW SUMMARY")
print("=" * 65)
print("Total quotations:", len(quotation_review))
print(
    "Quotations flagged for review:",
    int(quotation_review["Business_Rule_Review_Flag"].sum())
)
print(
    "Quotations with no triggered rules:",
    int((~quotation_review["Business_Rule_Review_Flag"]).sum())
)

print("\nSAMPLE QUOTATIONS FLAGGED FOR REVIEW")
display(
    quotation_review.loc[
        quotation_review["Business_Rule_Review_Flag"],
        [
            "Quote_ID",
            "Quote_Status",
            "Quote_Value",
            "Discount_Pct",
            "Revision_Count",
            "Engagement_Count",
            "Business_Rule_Flag_Count",
            "Review_Status",
        ],
    ].head(15)
)

BUSINESS-RULE REVIEW THRESHOLDS
95th percentile of quote value: 464749.25
High discount threshold: 20%
High revisions threshold: 4 or more
Zero engagement threshold: 0

RULE SUMMARY


,Rule,Flagged_Quotes
0,High_Discount_Review,68
1,High_Revisions_Review,55
2,Zero_Engagement_Review,30
3,High_Quote_Value_Review,15



OVERALL REVIEW SUMMARY
Total quotations: 300
Quotations flagged for review: 144
Quotations with no triggered rules: 156

SAMPLE QUOTATIONS FLAGGED FOR REVIEW


,Quote_ID,Quote_Status,Quote_Value,Discount_Pct,Revision_Count,Engagement_Count,Business_Rule_Flag_Count,Review_Status
0,SYN-Q-0001,Won,58286,20.66,0,9,1,Review recommended
1,SYN-Q-0002,Pending,390369,22.40,0,5,1,Review recommended
3,SYN-Q-0004,Pending,227856,13.85,4,8,1,Review recommended
11,SYN-Q-0012,Won,488177,22.90,0,8,2,Review recommended
13,SYN-Q-0014,Lost,384153,0.94,4,9,1,Review recommended
14,SYN-Q-0015,Won,362977,13.87,2,0,1,Review recommended
16,SYN-Q-0017,Won,263915,20.74,2,5,1,Review recommended
17,SYN-Q-0018,Won,77135,20.21,3,8,1,Review recommended
19,SYN-Q-0020,Lost,233437,23.82,3,4,1,Review recommended
20,SYN-Q-0021,Lost,257671,7.27,4,7,1,Review recommended


In [11]:
# Day 11: Step 10 - Save quotation anomaly outputs

# Output directory
output_dir = PROJECT_ROOT / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

# File 1: Complete quotation anomaly review table
quotation_output_file = (
    output_dir / "Day11_Quotation_Anomaly_Report.csv"
)

quotation_review.to_csv(
    quotation_output_file,
    index=False
)

# File 2: Detection rule summary
rule_summary_output_file = (
    output_dir / "Day11_Quotation_Anomaly_Rule_Summary.csv"
)

rule_summary.to_csv(
    rule_summary_output_file,
    index=False
)

# File 3: Overall summary
quotation_overall_summary = pd.DataFrame([
    {
        "Metric": "Total quotations",
        "Value": len(quotation_review),
    },
    {
        "Metric": "IQR flagged quotations",
        "Value": int(quotation_anomalies["Anomaly_Flag"].sum()),
    },
    {
        "Metric": "Business-rule flagged quotations",
        "Value": int(
            quotation_review["Business_Rule_Review_Flag"].sum()
        ),
    },
    {
        "Metric": "Quotations with no business-rule flags",
        "Value": int(
            (~quotation_review["Business_Rule_Review_Flag"]).sum()
        ),
    },
])

quotation_overall_file = (
    output_dir / "Day11_Quotation_Anomaly_Overall_Summary.csv"
)

quotation_overall_summary.to_csv(
    quotation_overall_file,
    index=False
)

# Verify the saved files
print("QUOTATION ANOMALY OUTPUT VALIDATION")
print("=" * 65)

for path in [
    quotation_output_file,
    rule_summary_output_file,
    quotation_overall_file,
]:
    print(f"{path.name}: {'FOUND' if path.exists() else 'MISSING'}")

saved_quotation_report = pd.read_csv(quotation_output_file)
saved_rule_summary = pd.read_csv(rule_summary_output_file)
saved_overall_summary = pd.read_csv(quotation_overall_file)

print("\nSaved quotation rows:", len(saved_quotation_report))
print("Saved rule summary rows:", len(saved_rule_summary))
print("Saved overall summary rows:", len(saved_overall_summary))

assert len(saved_quotation_report) == 300
assert saved_quotation_report["Quote_ID"].is_unique
assert len(saved_rule_summary) == 4
assert saved_quotation_report["Synthetic_Data_Flag"].eq(True).all()

print("\nPASS: Quotation anomaly outputs saved and verified.")

QUOTATION ANOMALY OUTPUT VALIDATION
Day11_Quotation_Anomaly_Report.csv: FOUND
Day11_Quotation_Anomaly_Rule_Summary.csv: FOUND
Day11_Quotation_Anomaly_Overall_Summary.csv: FOUND

Saved quotation rows: 300
Saved rule summary rows: 4
Saved overall summary rows: 4

PASS: Quotation anomaly outputs saved and verified.


In [12]:
# Day 11: Step 11.1 - Inspect project risk metrics

project_metrics = [
    "Delay_Days",
    "Effort_Overrun_Pct",
    "Resource_Utilization",
    "Forecast_Gross_Margin",
    "Change_Request_Count",
    "Dependency_Delay_Days",
    "Client_Delay_Days",
]

print("PROJECT RISK METRIC SUMMARY")
print("=" * 75)

display(
    project_risk[project_metrics]
    .describe()
    .round(2)
)

print("\nPROJECT RISK FLAGS")
print("=" * 75)

risk_flag_columns = [
    "Project_Delayed",
    "Effort_Overrun",
    "Low_Margin_Risk",
]

display(
    project_risk[risk_flag_columns]
    .apply(lambda column: column.value_counts(dropna=False))
    .fillna(0)
    .astype(int)
)

print("\nSYNTHETIC DATA CHECK")
print("=" * 75)

print(
    "All records marked synthetic:",
    project_risk["Synthetic_Data_Flag"].eq(True).all()
)

print(
    "Data-use labels:",
    project_risk["Data_Use"].unique().tolist()
)

PROJECT RISK METRIC SUMMARY


,Delay_Days,Effort_Overrun_Pct,Resource_Utilization,Forecast_Gross_Margin,Change_Request_Count,Dependency_Delay_Days,Client_Delay_Days
count,300.00,300.00,300.00,300.00,300.00,300.00,300.00
mean,20.65,0.11,0.81,0.28,1.24,5.56,3.96
std,10.96,0.16,0.14,0.10,1.09,3.65,2.64
min,-10.00,-0.32,0.55,0.10,0.00,0.00,0.00
25%,13.00,0.00,0.67,0.20,0.00,2.00,1.00
50%,21.00,0.11,0.81,0.28,1.00,5.00,4.00
75%,28.00,0.22,0.93,0.38,2.00,9.00,6.00
max,58.00,0.65,1.05,0.45,6.00,12.00,9.00



PROJECT RISK FLAGS


,Project_Delayed,Effort_Overrun,Low_Margin_Risk
0,38,143,221
1,262,157,79



SYNTHETIC DATA CHECK
All records marked synthetic: True
Data-use labels: ['Synthetic demonstration only']


In [13]:
# Day 11: Step 11.2 - Apply project anomaly review rules

project_anomalies = project_risk.copy()

# Provisional business-review rules
project_anomalies["High_Delay_Anomaly"] = (
    project_anomalies["Delay_Days"] >= 30
)

project_anomalies["High_Effort_Overrun_Anomaly"] = (
    project_anomalies["Effort_Overrun_Pct"] >= 0.20
)

project_anomalies["High_Resource_Utilization_Anomaly"] = (
    project_anomalies["Resource_Utilization"] >= 0.95
)

project_anomalies["Low_Forecast_Margin_Anomaly"] = (
    project_anomalies["Forecast_Gross_Margin"] < 0.15
)

project_anomalies["High_Change_Request_Anomaly"] = (
    project_anomalies["Change_Request_Count"] >= 4
)

project_anomalies["High_Dependency_Delay_Anomaly"] = (
    project_anomalies["Dependency_Delay_Days"] >= 9
)

project_anomalies["High_Client_Delay_Anomaly"] = (
    project_anomalies["Client_Delay_Days"] >= 7
)

# List of the new anomaly flags
project_anomaly_columns = [
    "High_Delay_Anomaly",
    "High_Effort_Overrun_Anomaly",
    "High_Resource_Utilization_Anomaly",
    "Low_Forecast_Margin_Anomaly",
    "High_Change_Request_Anomaly",
    "High_Dependency_Delay_Anomaly",
    "High_Client_Delay_Anomaly",
]

# Count triggered rules for each project
project_anomalies["Anomaly_Rule_Count"] = (
    project_anomalies[project_anomaly_columns]
    .sum(axis=1)
    .astype(int)
)

project_anomalies["Project_Anomaly_Flag"] = (
    project_anomalies["Anomaly_Rule_Count"] > 0
)

project_anomalies["Anomaly_Review_Status"] = np.where(
    project_anomalies["Project_Anomaly_Flag"],
    "Review recommended",
    "No proposed rule triggered"
)

project_anomalies["Anomaly_Data_Status"] = (
    "Synthetic demonstration; human review required"
)

# Summarize each rule
project_anomaly_rule_summary = pd.DataFrame({
    "Rule": project_anomaly_columns,
    "Projects_Flagged": [
        int(project_anomalies[column].sum())
        for column in project_anomaly_columns
    ],
    "Threshold_Type": ["Provisional business-review threshold"] * 7,
    "Data_Status": ["Synthetic demonstration only"] * 7,
})

print("PROJECT ANOMALY RULE SUMMARY")
print("=" * 75)
display(project_anomaly_rule_summary)

print("\nOVERALL PROJECT ANOMALY SUMMARY")
print("=" * 75)

print("Total projects:", len(project_anomalies))
print(
    "Projects flagged:",
    int(project_anomalies["Project_Anomaly_Flag"].sum())
)
print(
    "Projects not flagged:",
    int((~project_anomalies["Project_Anomaly_Flag"]).sum())
)
print(
    "Maximum anomaly rules triggered for one project:",
    int(project_anomalies["Anomaly_Rule_Count"].max())
)

print("\nSAMPLE FLAGGED PROJECTS")
print("=" * 75)

display(
    project_anomalies.loc[
        project_anomalies["Project_Anomaly_Flag"],
        [
            "Project_ID",
            "Delay_Days",
            "Effort_Overrun_Pct",
            "Resource_Utilization",
            "Forecast_Gross_Margin",
            "Change_Request_Count",
            "Dependency_Delay_Days",
            "Client_Delay_Days",
            "Anomaly_Rule_Count",
            "Anomaly_Review_Status",
        ],
    ].head(10)
)

# Validate results
assert len(project_anomalies) == len(project_risk)
assert project_anomalies["Project_ID"].is_unique
assert project_anomalies[project_anomaly_columns].notna().all().all()
assert project_anomalies["Synthetic_Data_Flag"].eq(True).all()

print("\nPASS: Project anomaly rules applied and validated.")

PROJECT ANOMALY RULE SUMMARY


,Rule,Projects_Flagged,Threshold_Type,Data_Status
0,High_Delay_Anomaly,62,Provisional business-review threshold,Synthetic demonstration only
1,High_Effort_Overrun_Anomaly,86,Provisional business-review threshold,Synthetic demonstration only
2,High_Resource_Utilization_Anomaly,61,Provisional business-review threshold,Synthetic demonstration only
3,Low_Forecast_Margin_Anomaly,46,Provisional business-review threshold,Synthetic demonstration only
4,High_Change_Request_Anomaly,8,Provisional business-review threshold,Synthetic demonstration only
5,High_Dependency_Delay_Anomaly,81,Provisional business-review threshold,Synthetic demonstration only
6,High_Client_Delay_Anomaly,71,Provisional business-review threshold,Synthetic demonstration only



OVERALL PROJECT ANOMALY SUMMARY
Total projects: 300
Projects flagged: 203
Projects not flagged: 97
Maximum anomaly rules triggered for one project: 6

SAMPLE FLAGGED PROJECTS


,Project_ID,Delay_Days,Effort_Overrun_Pct,Resource_Utilization,Forecast_Gross_Margin,Change_Request_Count,Dependency_Delay_Days,Client_Delay_Days,Anomaly_Rule_Count,Anomaly_Review_Status
0,PRJ-0001,16,0.1580,0.683,0.1181,0,10,6,2,Review recommended
1,PRJ-0002,8,0.1980,0.687,0.4180,0,9,5,1,Review recommended
2,PRJ-0003,17,-0.0371,0.677,0.3358,0,2,8,1,Review recommended
3,PRJ-0004,34,-0.0312,0.618,0.3679,1,10,8,3,Review recommended
4,PRJ-0005,32,0.1482,1.022,0.3124,2,0,6,2,Review recommended
6,PRJ-0007,36,-0.0716,0.662,0.2840,1,8,5,1,Review recommended
7,PRJ-0008,11,0.3534,0.973,0.2502,1,3,3,2,Review recommended
8,PRJ-0009,18,0.1027,0.750,0.3400,0,9,6,1,Review recommended
9,PRJ-0010,29,0.1596,0.568,0.4005,0,10,7,2,Review recommended
11,PRJ-0012,26,0.0400,0.870,0.1491,3,0,4,1,Review recommended



PASS: Project anomaly rules applied and validated.


In [17]:
# Day 11: Step 11.3 - Save project anomaly outputs

output_dir = PROJECT_ROOT / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

# 1. Save the detailed anomaly report
project_anomaly_report_file = (
    output_dir / "Day11_Project_Anomaly_Report.csv"
)

project_anomalies.to_csv(
    project_anomaly_report_file,
    index=False
)

# 2. Save the rule summary
project_anomaly_rule_summary_file = (
    output_dir / "Day11_Project_Anomaly_Rule_Summary.csv"
)

project_anomaly_rule_summary.to_csv(
    project_anomaly_rule_summary_file,
    index=False
)

# 3. Save the overall summary
project_anomaly_overall_summary = pd.DataFrame([
    {
        "Metric": "Total projects",
        "Value": len(project_anomalies),
    },
    {
        "Metric": "Projects flagged",
        "Value": int(
            project_anomalies["Project_Anomaly_Flag"].sum()
        ),
    },
    {
        "Metric": "Projects not flagged",
        "Value": int(
            (~project_anomalies["Project_Anomaly_Flag"]).sum()
        ),
    },
    {
        "Metric": "Maximum anomaly rules triggered",
        "Value": int(
            project_anomalies["Anomaly_Rule_Count"].max()
        ),
    },
])

project_anomaly_overall_file = (
    output_dir / "Day11_Project_Anomaly_Overall_Summary.csv"
)

project_anomaly_overall_summary.to_csv(
    project_anomaly_overall_file,
    index=False
)

# 4. Reload and verify the saved files
saved_project_report = pd.read_csv(
    project_anomaly_report_file
)

saved_project_rule_summary = pd.read_csv(
    project_anomaly_rule_summary_file
)

saved_project_overall_summary = pd.read_csv(
    project_anomaly_overall_file
)

print("PROJECT ANOMALY OUTPUT VALIDATION")
print("=" * 65)

for file_path in [
    project_anomaly_report_file,
    project_anomaly_rule_summary_file,
    project_anomaly_overall_file,
]:
    print(
        f"{file_path.name}:",
        "FOUND" if file_path.exists() else "MISSING"
    )

print("\nSaved project rows:", len(saved_project_report))
print("Saved rule summary rows:", len(saved_project_rule_summary))
print("Saved overall summary rows:", len(saved_project_overall_summary))

assert len(saved_project_report) == 300
assert saved_project_report["Project_ID"].is_unique
assert len(saved_project_rule_summary) == 7
assert len(saved_project_overall_summary) == 4
assert saved_project_report["Synthetic_Data_Flag"].eq(True).all()

print("\nPASS: Project anomaly outputs saved and verified.")

PROJECT ANOMALY OUTPUT VALIDATION
Day11_Project_Anomaly_Report.csv: FOUND
Day11_Project_Anomaly_Rule_Summary.csv: FOUND
Day11_Project_Anomaly_Overall_Summary.csv: FOUND

Saved project rows: 300
Saved rule summary rows: 7
Saved overall summary rows: 4

PASS: Project anomaly outputs saved and verified.


In [23]:
# Day 11: Finance Anomaly Detection
# Synthetic demonstration only; these are not actual NexziByte financial results.

required_columns = {"Month", "Revenue", "Synthetic_Data_Flag", "Data_Use"}
missing_columns = required_columns - set(monthly_revenue.columns)
assert not missing_columns, f"Missing revenue columns: {sorted(missing_columns)}"

finance_original = monthly_revenue.copy(deep=True)
finance_anomalies = monthly_revenue.copy(deep=True)

finance_anomalies["Month"] = pd.to_datetime(
    finance_anomalies["Month"], errors="raise"
)
finance_anomalies["Revenue"] = pd.to_numeric(
    finance_anomalies["Revenue"], errors="raise"
)
finance_anomalies = finance_anomalies.sort_values("Month").reset_index(drop=True)

assert not finance_anomalies["Month"].duplicated().any(), "Duplicate months found."
assert finance_anomalies["Month"].notna().all(), "Missing month values found."
assert finance_anomalies["Revenue"].notna().all(), "Missing revenue values found."
assert (finance_anomalies["Revenue"] >= 0).all(), "Negative revenue values require review."

def finance_iqr_bounds(series):
    values = sorted(pd.Series(series).dropna().astype(float).tolist())
    if len(values) < 4:
        raise ValueError("At least four valid observations are required for IQR bounds.")
    q1 = values[int(np.floor((len(values) - 1) * 0.25))]
    q3 = values[int(np.floor((len(values) - 1) * 0.75))]
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

finance_anomalies["Previous_Month_Revenue"] = finance_anomalies["Revenue"].shift(1)
finance_anomalies["Month_Over_Month_Change_Pct"] = (
    finance_anomalies["Revenue"].pct_change(fill_method=None) * 100
).round(2)

revenue_lower, revenue_upper = finance_iqr_bounds(finance_anomalies["Revenue"])
change_lower, change_upper = finance_iqr_bounds(
    finance_anomalies["Month_Over_Month_Change_Pct"]
)

finance_anomalies["Revenue_IQR_Anomaly_Flag"] = (
    (finance_anomalies["Revenue"] < revenue_lower) |
    (finance_anomalies["Revenue"] > revenue_upper)
)
finance_anomalies["Revenue_Change_IQR_Anomaly_Flag"] = (
    finance_anomalies["Month_Over_Month_Change_Pct"].notna() &
    (
        (finance_anomalies["Month_Over_Month_Change_Pct"] < change_lower) |
        (finance_anomalies["Month_Over_Month_Change_Pct"] > change_upper)
    )
)
finance_anomalies["Finance_Anomaly_Flag"] = (
    finance_anomalies["Revenue_IQR_Anomaly_Flag"] |
    finance_anomalies["Revenue_Change_IQR_Anomaly_Flag"]
)

def explain_finance_anomaly(row):
    reasons = []
    if row["Revenue_IQR_Anomaly_Flag"]:
        reasons.append(
            f"Revenue {row['Revenue']:.2f} is outside the provisional "
            f"IQR bounds ({revenue_lower:.2f} to {revenue_upper:.2f})."
        )
    if row["Revenue_Change_IQR_Anomaly_Flag"]:
        change = row["Month_Over_Month_Change_Pct"]
        reasons.append(
            f"Month-to-month revenue change of {change:.2f}% is outside "
            f"the provisional IQR bounds ({change_lower:.2f}% to {change_upper:.2f}%)."
        )
    return "; ".join(reasons) if reasons else (
        "No configured IQR threshold was breached; this does not establish that revenue is normal."
    )

finance_anomalies["Anomaly_Rules_Triggered"] = finance_anomalies.apply(
    lambda row: "; ".join(
        rule for rule, flagged in [
            ("Unusual_Monthly_Revenue", row["Revenue_IQR_Anomaly_Flag"]),
            ("Unusual_Monthly_Revenue_Change", row["Revenue_Change_IQR_Anomaly_Flag"]),
        ] if flagged
    ) or "None",
    axis=1,
)
finance_anomalies["Anomaly_Reason"] = finance_anomalies.apply(
    explain_finance_anomaly, axis=1
)
finance_anomalies["Review_Status"] = np.where(
    finance_anomalies["Finance_Anomaly_Flag"],
    "Review recommended",
    "No IQR flag; routine monitoring",
)
finance_anomalies["Data_Status"] = (
    "Synthetic demonstration data; not actual NexziByte financial results"
)
finance_anomalies["Month"] = finance_anomalies["Month"].dt.strftime("%Y-%m")
finance_anomalies["Previous_Month_Revenue"] = (
    finance_anomalies["Previous_Month_Revenue"].round(2)
)
finance_anomalies["Revenue"] = finance_anomalies["Revenue"].round(2)

processed_dir = PROJECT_ROOT / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

finance_report_path = processed_dir / "Day11_Finance_Anomaly_Report.csv"
finance_rule_summary_path = processed_dir / "Day11_Finance_Anomaly_Rule_Summary.csv"
finance_overall_path = processed_dir / "Day11_Finance_Anomaly_Overall_Summary.csv"

finance_anomalies.to_csv(finance_report_path, index=False)

finance_rule_summary = pd.DataFrame([
    {
        "Rule": "Revenue_IQR_Anomaly_Flag",
        "Records_Flagged": int(finance_anomalies["Revenue_IQR_Anomaly_Flag"].sum()),
        "Threshold_Type": f"Below {revenue_lower:.2f} or above {revenue_upper:.2f}; provisional IQR bounds",
        "Data_Status": "Synthetic demonstration data; business approval required",
    },
    {
        "Rule": "Revenue_Change_IQR_Anomaly_Flag",
        "Records_Flagged": int(finance_anomalies["Revenue_Change_IQR_Anomaly_Flag"].sum()),
        "Threshold_Type": f"Below {change_lower:.2f}% or above {change_upper:.2f}%; provisional IQR bounds",
        "Data_Status": "Synthetic demonstration data; business approval required",
    },
])
finance_rule_summary.to_csv(finance_rule_summary_path, index=False)

finance_overall_summary = pd.DataFrame([{
    "Total_Months": int(len(finance_anomalies)),
    "Revenue_Amount_IQR_Flags": int(finance_anomalies["Revenue_IQR_Anomaly_Flag"].sum()),
    "Month_Change_IQR_Flags": int(finance_anomalies["Revenue_Change_IQR_Anomaly_Flag"].sum()),
    "Months_With_Any_Anomaly": int(finance_anomalies["Finance_Anomaly_Flag"].sum()),
    "Months_Without_Anomaly": int((~finance_anomalies["Finance_Anomaly_Flag"]).sum()),
    "Synthetic_Records": int(finance_anomalies["Synthetic_Data_Flag"].sum()),
    "Revenue_Lower_Fence": round(float(revenue_lower), 2),
    "Revenue_Upper_Fence": round(float(revenue_upper), 2),
    "Change_Lower_Fence_Pct": round(float(change_lower), 2),
    "Change_Upper_Fence_Pct": round(float(change_upper), 2),
    "Data_Status": "Synthetic demonstration data; not actual NexziByte financial results",
}])
finance_overall_summary.to_csv(finance_overall_path, index=False)

assert len(finance_anomalies) == len(finance_original), "Unexpected change in row count."
assert finance_anomalies["Month"].is_unique, "Output months are not unique."
assert monthly_revenue.equals(finance_original), "Source DataFrame was unexpectedly modified."
assert all(p.exists() and p.stat().st_size > 0 for p in [
    finance_report_path, finance_rule_summary_path, finance_overall_path
]), "One or more finance outputs are missing or empty."

print("Finance anomaly analysis cell completed successfully.")
print(f"Months analysed: {len(finance_anomalies)}")
print(f"Revenue IQR bounds: {revenue_lower:.2f} to {revenue_upper:.2f}")
print(f"Monthly change IQR bounds: {change_lower:.2f}% to {change_upper:.2f}%")
print(f"Months flagged: {int(finance_anomalies['Finance_Anomaly_Flag'].sum())}")
print("Flagged months:")
print(finance_anomalies.loc[
    finance_anomalies["Finance_Anomaly_Flag"],
    ["Month", "Revenue", "Month_Over_Month_Change_Pct", "Anomaly_Reason"]
].to_string(index=False))
print("Created:")
for output_path in [finance_report_path, finance_rule_summary_path, finance_overall_path]:
    print(f"- {output_path.name} ({output_path.stat().st_size} bytes)")


Finance anomaly analysis cell completed successfully.
Months analysed: 36
Revenue IQR bounds: 361704.50 to 898707.62
Monthly change IQR bounds: -13.40% to 16.36%
Months flagged: 1
Flagged months:
  Month   Revenue  Month_Over_Month_Change_Pct                                                                                      Anomaly_Reason
2023-05 516770.21                       -14.19 Month-to-month revenue change of -14.19% is outside the provisional IQR bounds (-13.40% to 16.36%).
Created:
- Day11_Finance_Anomaly_Report.csv (10486 bytes)
- Day11_Finance_Anomaly_Rule_Summary.csv (339 bytes)
- Day11_Finance_Anomaly_Overall_Summary.csv (341 bytes)


In [25]:

import pandas as pd

# Day 11: Step 11.4 - Create business alert rules

alert_rules = pd.DataFrame([
    {
        "Alert_ID": "ALT-001",
        "Alert_Name": "Significant Project Delay",
        "Data_Source": "Project Risk Dataset",
        "Trigger_Condition": "Delay_Days >= 30",
        "Severity": "High",
        "Recommended_Action": "Review the project schedule, blockers, and recovery plan.",
        "Escalation": "Notify the project manager for review.",
    },
    {
        "Alert_ID": "ALT-002",
        "Alert_Name": "High Effort Overrun",
        "Data_Source": "Project Risk Dataset",
        "Trigger_Condition": "Effort_Overrun_Pct >= 0.20",
        "Severity": "High",
        "Recommended_Action": "Compare planned versus actual effort and investigate the variance.",
        "Escalation": "Notify the project manager and delivery lead.",
    },
    {
        "Alert_ID": "ALT-003",
        "Alert_Name": "High Resource Utilization",
        "Data_Source": "Project Risk Dataset",
        "Trigger_Condition": "Resource_Utilization >= 0.95",
        "Severity": "High",
        "Recommended_Action": "Review team capacity, workload, and resource allocation.",
        "Escalation": "Notify the delivery lead for capacity review.",
    },
    {
        "Alert_ID": "ALT-004",
        "Alert_Name": "Low Forecast Gross Margin",
        "Data_Source": "Project Risk Dataset",
        "Trigger_Condition": "Forecast_Gross_Margin < 0.15",
        "Severity": "High",
        "Recommended_Action": "Review forecast costs, project scope, and margin assumptions.",
        "Escalation": "Refer to the project and finance leads.",
    },
    {
        "Alert_ID": "ALT-005",
        "Alert_Name": "Multiple Change Requests",
        "Data_Source": "Project Risk Dataset",
        "Trigger_Condition": "Change_Request_Count >= 4",
        "Severity": "Medium",
        "Recommended_Action": "Review scope changes and their delivery and cost impact.",
        "Escalation": "Request a scope review from the project manager.",
    },
    {
        "Alert_ID": "ALT-006",
        "Alert_Name": "Dependency Delay",
        "Data_Source": "Project Risk Dataset",
        "Trigger_Condition": "Dependency_Delay_Days >= 9",
        "Severity": "High",
        "Recommended_Action": "Identify blocked dependencies and agree on mitigation actions.",
        "Escalation": "Notify the project manager and dependency owner.",
    },
    {
        "Alert_ID": "ALT-007",
        "Alert_Name": "Client-Related Delay",
        "Data_Source": "Project Risk Dataset",
        "Trigger_Condition": "Client_Delay_Days >= 7",
        "Severity": "Medium",
        "Recommended_Action": "Review outstanding client decisions, approvals, or deliverables.",
        "Escalation": "Request follow-up from the client-facing owner.",
    },
    {
        "Alert_ID": "ALT-008",
        "Alert_Name": "High Quotation Discount",
        "Data_Source": "Quotation Dataset",
        "Trigger_Condition": "Discount_Pct >= 20",
        "Severity": "Medium",
        "Recommended_Action": "Review discount justification and commercial approval.",
        "Escalation": "Refer to the sales or commercial approver.",
    },
    {
        "Alert_ID": "ALT-009",
        "Alert_Name": "Repeated Quotation Revisions",
        "Data_Source": "Quotation Dataset",
        "Trigger_Condition": "Revision_Count >= 4",
        "Severity": "Medium",
        "Recommended_Action": "Review revision history and clarify customer requirements.",
        "Escalation": "Request a quotation review from the sales owner.",
    },
    {
        "Alert_ID": "ALT-010",
        "Alert_Name": "Quotation Without Engagement",
        "Data_Source": "Quotation Dataset",
        "Trigger_Condition": "Engagement_Count == 0",
        "Severity": "Medium",
        "Recommended_Action": "Check follow-up history and decide whether outreach is appropriate.",
        "Escalation": "Refer to the quotation owner.",
    },
    {
        "Alert_ID": "ALT-011",
        "Alert_Name": "High-Value Quotation Review",
        "Data_Source": "Quotation Dataset",
        "Trigger_Condition": "Quote_Value >= dataset 95th percentile",
        "Severity": "Medium",
        "Recommended_Action": "Review commercial assumptions and approval requirements.",
        "Escalation": "Refer to the authorized commercial approver.",
    },
    {
        "Alert_ID": "ALT-012",
        "Alert_Name": "Finance Revenue Movement Review",
        "Data_Source": "Monthly Revenue Dataset",
        "Trigger_Condition": (
            "Month-over-month revenue change falls outside "
            "dataset-derived provisional IQR fences"
        ),
        "Severity": "Medium",
        "Recommended_Action": (
            "Review reporting context, possible seasonality, "
            "and underlying revenue records."
        ),
        "Escalation": "Refer to the finance owner for review.",
    },
])

# Add governance and data-status information.
alert_rules["Threshold_Status"] = (
    "Provisional; business approval required"
)

alert_rules["Data_Status"] = (
    "Rules demonstrated using synthetic data; not a live alert system"
)

# Display the alert rules.
print("BUSINESS ALERT RULES")
print("=" * 65)

display(alert_rules)

# --------------------------------------------------
# VALIDATION CHECKS
# --------------------------------------------------

# Confirm all 12 alert rules exist.
assert len(alert_rules) == 12, (
    f"Expected 12 alert rules, found {len(alert_rules)}"
)

# Confirm alert IDs are unique.
assert alert_rules["Alert_ID"].is_unique, (
    "Duplicate Alert_ID values detected."
)

# Confirm IDs cover ALT-001 through ALT-012.
expected_ids = [
    f"ALT-{number:03d}"
    for number in range(1, 13)
]

assert sorted(alert_rules["Alert_ID"].tolist()) == expected_ids, (
    "Alert IDs are missing or unexpected."
)

# Confirm required fields are populated.
required_columns = [
    "Alert_ID",
    "Alert_Name",
    "Data_Source",
    "Trigger_Condition",
    "Severity",
    "Recommended_Action",
    "Escalation",
    "Threshold_Status",
    "Data_Status",
]

assert alert_rules[required_columns].notna().all().all(), (
    "Missing values detected in required alert-rule fields."
)

# Confirm severity values are valid.
assert alert_rules["Severity"].isin(
    ["High", "Medium", "Low"]
).all(), "Invalid severity value detected."

# Confirm alert names are unique.
assert alert_rules["Alert_Name"].is_unique, (
    "Duplicate alert names detected."
)

# --------------------------------------------------
# VALIDATION SUMMARY
# --------------------------------------------------

print("\nVALIDATION SUMMARY")
print("=" * 65)

print("Total alert rules:", len(alert_rules))
print("Unique alert IDs:", alert_rules["Alert_ID"].nunique())
print("Unique alert names:", alert_rules["Alert_Name"].nunique())

print("\nSeverity counts:")

severity_counts = (
    alert_rules["Severity"]
    .value_counts()
    .reindex(["High", "Medium", "Low"], fill_value=0)
    .rename_axis("Severity")
    .to_frame("Rule_Count")
)

display(severity_counts)

print("\nData status:")
print(alert_rules["Data_Status"].iloc[0])

print("\nThreshold status:")
print(alert_rules["Threshold_Status"].iloc[0])

print("\nPASS: All 12 business alert rules created and validated.")


BUSINESS ALERT RULES


,Alert_ID,Alert_Name,Data_Source,Trigger_Condition,Severity,Recommended_Action,Escalation,Threshold_Status,Data_Status
0,ALT-001,Significant Project Delay,Project Risk Dataset,Delay_Days >= 30,High,"Review the project schedule, blockers, and rec...",Notify the project manager for review.,Provisional; business approval required,Rules demonstrated using synthetic data; not a...
1,ALT-002,High Effort Overrun,Project Risk Dataset,Effort_Overrun_Pct >= 0.20,High,Compare planned versus actual effort and inves...,Notify the project manager and delivery lead.,Provisional; business approval required,Rules demonstrated using synthetic data; not a...
2,ALT-003,High Resource Utilization,Project Risk Dataset,Resource_Utilization >= 0.95,High,"Review team capacity, workload, and resource a...",Notify the delivery lead for capacity review.,Provisional; business approval required,Rules demonstrated using synthetic data; not a...
3,ALT-004,Low Forecast Gross Margin,Project Risk Dataset,Forecast_Gross_Margin < 0.15,High,"Review forecast costs, project scope, and marg...",Refer to the project and finance leads.,Provisional; business approval required,Rules demonstrated using synthetic data; not a...
4,ALT-005,Multiple Change Requests,Project Risk Dataset,Change_Request_Count >= 4,Medium,Review scope changes and their delivery and co...,Request a scope review from the project manager.,Provisional; business approval required,Rules demonstrated using synthetic data; not a...
5,ALT-006,Dependency Delay,Project Risk Dataset,Dependency_Delay_Days >= 9,High,Identify blocked dependencies and agree on mit...,Notify the project manager and dependency owner.,Provisional; business approval required,Rules demonstrated using synthetic data; not a...
6,ALT-007,Client-Related Delay,Project Risk Dataset,Client_Delay_Days >= 7,Medium,"Review outstanding client decisions, approvals...",Request follow-up from the client-facing owner.,Provisional; business approval required,Rules demonstrated using synthetic data; not a...
7,ALT-008,High Quotation Discount,Quotation Dataset,Discount_Pct >= 20,Medium,Review discount justification and commercial a...,Refer to the sales or commercial approver.,Provisional; business approval required,Rules demonstrated using synthetic data; not a...
8,ALT-009,Repeated Quotation Revisions,Quotation Dataset,Revision_Count >= 4,Medium,Review revision history and clarify customer r...,Request a quotation review from the sales owner.,Provisional; business approval required,Rules demonstrated using synthetic data; not a...
9,ALT-010,Quotation Without Engagement,Quotation Dataset,Engagement_Count == 0,Medium,Check follow-up history and decide whether out...,Refer to the quotation owner.,Provisional; business approval required,Rules demonstrated using synthetic data; not a...



VALIDATION SUMMARY
Total alert rules: 12
Unique alert IDs: 12
Unique alert names: 12

Severity counts:


,Rule_Count
Severity,
High,5
Medium,7
Low,0



Data status:
Rules demonstrated using synthetic data; not a live alert system

Threshold status:
Provisional; business approval required

PASS: All 12 business alert rules created and validated.


In [20]:
# Day 11: Step 11.5 - Save and verify business alert rules

output_dir = PROJECT_ROOT / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

alert_rules_file = output_dir / "Day11_Business_Alert_Rules.csv"

# Save the rules
alert_rules.to_csv(alert_rules_file, index=False)

# Reload the saved file for verification
saved_alert_rules = pd.read_csv(alert_rules_file)

print("BUSINESS ALERT RULES VALIDATION")
print("=" * 65)

print(
    "File:",
    alert_rules_file.name,
    "|",
    "FOUND" if alert_rules_file.exists() else "MISSING"
)

print("Saved alert rules:", len(saved_alert_rules))
print("Unique alert IDs:", saved_alert_rules["Alert_ID"].nunique())

print("\nSeverity counts:")
display(
    saved_alert_rules["Severity"]
    .value_counts()
    .rename_axis("Severity")
    .to_frame("Rule_Count")
)

# Validation checks
assert alert_rules_file.exists()
assert len(saved_alert_rules) == 11
assert saved_alert_rules["Alert_ID"].is_unique
assert saved_alert_rules["Trigger_Condition"].notna().all()
assert saved_alert_rules["Recommended_Action"].notna().all()
assert saved_alert_rules["Threshold_Status"].eq(
    "Provisional; business approval required"
).all()

print("\nPASS: Business alert rules saved and verified.")

BUSINESS ALERT RULES VALIDATION
File: Day11_Business_Alert_Rules.csv | FOUND
Saved alert rules: 11
Unique alert IDs: 11

Severity counts:


,Rule_Count
Severity,
Medium,6
High,5



PASS: Business alert rules saved and verified.


In [21]:
# Day 11: Step 11.6 - Audit all required output files

processed_dir = PROJECT_ROOT / "data" / "processed"

required_outputs = [
    "Day11_Customer_Recommendations.csv",
    "Day11_Recommendation_Summary.csv",
    "Day11_Quotation_Anomaly_Report.csv",
    "Day11_Quotation_Anomaly_Rule_Summary.csv",
    "Day11_Quotation_Anomaly_Overall_Summary.csv",
    "Day11_Project_Anomaly_Report.csv",
    "Day11_Project_Anomaly_Rule_Summary.csv",
    "Day11_Project_Anomaly_Overall_Summary.csv",
    "Day11_Business_Alert_Rules.csv",
]

audit_results = []

for filename in required_outputs:
    file_path = processed_dir / filename

    audit_results.append({
        "File_Name": filename,
        "Exists": file_path.exists(),
        "File_Size_Bytes": (
            file_path.stat().st_size
            if file_path.exists()
            else 0
        ),
    })

day11_file_audit = pd.DataFrame(audit_results)

print("DAY 11 REQUIRED FILE AUDIT")
print("=" * 70)
display(day11_file_audit)

print("\nFiles found:", int(day11_file_audit["Exists"].sum()))
print("Files required:", len(required_outputs))

assert day11_file_audit["Exists"].all()

print("\nPASS: All required Day 11 output files exist.")

DAY 11 REQUIRED FILE AUDIT


,File_Name,Exists,File_Size_Bytes
0,Day11_Customer_Recommendations.csv,True,122239
1,Day11_Recommendation_Summary.csv,True,263
2,Day11_Quotation_Anomaly_Report.csv,True,74779
3,Day11_Quotation_Anomaly_Rule_Summary.csv,True,127
4,Day11_Quotation_Anomaly_Overall_Summary.csv,True,144
5,Day11_Project_Anomaly_Report.csv,True,88293
6,Day11_Project_Anomaly_Rule_Summary.csv,True,739
7,Day11_Project_Anomaly_Overall_Summary.csv,True,116
8,Day11_Business_Alert_Rules.csv,True,3368



Files found: 9
Files required: 9

PASS: All required Day 11 output files exist.


In [22]:
# Day 11: Step 11.7 - Create the final report

from datetime import datetime

reports_dir = PROJECT_ROOT / "reports"
processed_dir = PROJECT_ROOT / "data" / "processed"

reports_dir.mkdir(parents=True, exist_ok=True)

report_file = (
    reports_dir / "Day11_Recommendation_Anomaly_Detection_Report.md"
)

# Load saved outputs so the report reflects the files on disk.
recommendations_df = pd.read_csv(
    processed_dir / "Day11_Customer_Recommendations.csv"
)

recommendation_summary_df = pd.read_csv(
    processed_dir / "Day11_Recommendation_Summary.csv"
)

quotation_report_df = pd.read_csv(
    processed_dir / "Day11_Quotation_Anomaly_Report.csv"
)

quotation_rules_df = pd.read_csv(
    processed_dir / "Day11_Quotation_Anomaly_Rule_Summary.csv"
)

quotation_overall_df = pd.read_csv(
    processed_dir / "Day11_Quotation_Anomaly_Overall_Summary.csv"
)

project_report_df = pd.read_csv(
    processed_dir / "Day11_Project_Anomaly_Report.csv"
)

project_rules_df = pd.read_csv(
    processed_dir / "Day11_Project_Anomaly_Rule_Summary.csv"
)

project_overall_df = pd.read_csv(
    processed_dir / "Day11_Project_Anomaly_Overall_Summary.csv"
)

alert_rules_df = pd.read_csv(
    processed_dir / "Day11_Business_Alert_Rules.csv"
)

# Derive counts from saved files.
recommendation_count = len(recommendations_df)

quotation_count = len(quotation_report_df)
quotation_iqr_count = int(
    quotation_report_df["Anomaly_Flag"].sum()
)
quotation_business_flag_count = int(
    quotation_report_df["Business_Rule_Review_Flag"].sum()
)
quotation_no_flag_count = (
    quotation_count - quotation_business_flag_count
)

project_count = len(project_report_df)
project_flagged_count = int(
    project_report_df["Project_Anomaly_Flag"].sum()
)
project_not_flagged_count = (
    project_count - project_flagged_count
)

# Format rule summaries.
quotation_rule_lines = "\n".join(
    f"| {row['Rule']} | {int(row['Flagged_Quotes'])} |"
    for _, row in quotation_rules_df.iterrows()
)

project_rule_lines = "\n".join(
    f"| {row['Rule']} | {int(row['Projects_Flagged'])} |"
    for _, row in project_rules_df.iterrows()
)

recommendation_action_counts = (
    recommendations_df["Recommended_Next_Action"]
    .value_counts()
    if "Recommended_Next_Action" in recommendations_df.columns
    else recommendations_df["Next_Best_Action"].value_counts()
    if "Next_Best_Action" in recommendations_df.columns
    else recommendations_df["Recommended_Action"].value_counts()
    if "Recommended_Action" in recommendations_df.columns
    else None
)

if recommendation_action_counts is not None:
    recommendation_lines = "\n".join(
        f"| {action} | {int(count)} |"
        for action, count in recommendation_action_counts.items()
    )
else:
    recommendation_lines = (
        "See `Day11_Recommendation_Summary.csv` for the saved "
        "recommendation summary."
    )

report_content = f"""# Day 11: Recommendation and Anomaly Detection Report

## 1. Executive Summary

Day 11 implemented a rule-based customer recommendation engine,
quotation anomaly screening, project-risk anomaly screening, and a
business alert-rules catalogue.

All nine required CSV deliverables were present during the final
file audit. The outputs were generated using synthetic demonstration
data and provisional business-review rules.

**Status:** Technical deliverables completed; business validation and
threshold approval remain pending.

## 2. Scope and Objectives

- Recommend a next action for each customer profile.
- Screen quotation records for unusual values and business-rule conditions.
- Screen project records for potential delivery, effort, resource,
  margin, and delay concerns.
- Define alert conditions, severity, recommended action, and escalation.
- Save outputs and verify that required files exist.

## 3. Customer Recommendation Engine

The recommendation engine produced **{recommendation_count} customer
recommendation records**.

The rule-based actions include:
- Explore expansion opportunities.
- Follow up on pending quotations.
- Review customer re-engagement.
- Explore relevant follow-on services.
- Review customer needs and quotation history.

### Recommendation action counts

{recommendation_lines}

Recommendations are illustrative and depend on the available synthetic
quotation and customer-profile fields. They are not verified commercial
recommendations for real customers.

## 4. Quotation Anomaly Detection

The quotation anomaly report contains **{quotation_count} records**.

### Statistical screening

The interquartile-range (IQR) screening identified
**{quotation_iqr_count} quotations** as statistical outliers across the
four inspected metrics: quote value, discount percentage, revision count,
and engagement count.

No IQR flags does not prove that all records are normal or correct.
The IQR method only identifies observations outside its calculated
bounds.

### Provisional business-rule screening

- Quotations flagged by at least one business rule:
  **{quotation_business_flag_count}**
- Quotations with no business-rule flags:
  **{quotation_no_flag_count}**

| Business rule | Quotations flagged |
|---|---:|
{quotation_rule_lines}

The business rules screen for high discounts, repeated revisions,
zero engagement, and high quotation values. Rule counts can overlap;
they must not be summed to calculate unique flagged quotations.

A review flag is not evidence of fraud, error, or a lost sale. All
thresholds are provisional and all quotation records are synthetic.

## 5. Project Risk Anomaly Detection

The project anomaly report contains **{project_count} synthetic project
records**.

- Projects flagged by at least one proposed rule:
  **{project_flagged_count}**
- Projects with no proposed rule triggered:
  **{project_not_flagged_count}**

| Project anomaly rule | Projects flagged |
|---|---:|
{project_rule_lines}

The proposed rules screen for:
- Significant delivery delays.
- High effort overruns.
- High resource utilization.
- Low forecast gross margin.
- Multiple change requests.
- Dependency delays.
- Client-related delays.

Counts overlap because a project can trigger multiple rules. These new
anomaly flags are distinct from the existing Day 9 project-risk labels.

The thresholds are provisional business-review criteria, not approved
company policies. Flagged projects require human review.

## 6. Business Alert Rules

The alert-rules catalogue contains **{len(alert_rules_df)} rules**:
**{int(alert_rules_df['Severity'].eq('High').sum())} High** and
**{int(alert_rules_df['Severity'].eq('Medium').sum())} Medium**.

The catalogue includes:
- Alert ID and name.
- Data source and trigger condition.
- Severity.
- Recommended action.
- Escalation path.
- Threshold approval status.
- Synthetic-data limitation.

These rows define proposed alert conditions; they do not implement
live monitoring, automated notifications, or escalation workflows.
Business owners must approve thresholds and escalation responsibilities
before operational use.

## 7. Deliverables

The following nine files were present during the final audit:

| File | Purpose |
|---|---|
| `Day11_Customer_Recommendations.csv` | Customer-level next-action recommendations |
| `Day11_Recommendation_Summary.csv` | Recommendation summary |
| `Day11_Quotation_Anomaly_Report.csv` | Detailed quotation review flags |
| `Day11_Quotation_Anomaly_Rule_Summary.csv` | Quotation rule counts |
| `Day11_Quotation_Anomaly_Overall_Summary.csv` | Overall quotation screening counts |
| `Day11_Project_Anomaly_Report.csv` | Detailed project review flags |
| `Day11_Project_Anomaly_Rule_Summary.csv` | Project rule counts |
| `Day11_Project_Anomaly_Overall_Summary.csv` | Overall project screening counts |
| `Day11_Business_Alert_Rules.csv` | Proposed business alert catalogue |

## 8. Limitations and Business Validation

1. The quotation and project datasets are synthetic demonstration data.
2. The customer recommendations are rule-based, not experimentally
   validated or proven to increase revenue.
3. Anomaly flags indicate review candidates, not confirmed errors,
   fraud, or operational failures.
4. The thresholds require review and approval by the relevant business
   stakeholders.
5. The alert catalogue does not send live notifications.
6. Real operational use requires validated source data, approved
   definitions, appropriate access controls, and monitoring.
7. Business impact and predictive performance have not been established
   by this demonstration.

## 9. Final Status

**Technical status:** Day 11 outputs generated and required-file audit
passed.

**Business status:** Validation pending. Threshold approval and review
of recommendations remain outstanding.

**Data status:** Synthetic demonstration data; not verified real-business
results.
"""

# Save the report.
report_file.write_text(report_content, encoding="utf-8")

# Verify the report and all nine required output files.
required_output_names = [
    "Day11_Customer_Recommendations.csv",
    "Day11_Recommendation_Summary.csv",
    "Day11_Quotation_Anomaly_Report.csv",
    "Day11_Quotation_Anomaly_Rule_Summary.csv",
    "Day11_Quotation_Anomaly_Overall_Summary.csv",
    "Day11_Project_Anomaly_Report.csv",
    "Day11_Project_Anomaly_Rule_Summary.csv",
    "Day11_Project_Anomaly_Overall_Summary.csv",
    "Day11_Business_Alert_Rules.csv",
]

missing_outputs = [
    name for name in required_output_names
    if not (processed_dir / name).exists()
]

saved_report_text = report_file.read_text(encoding="utf-8")

print("DAY 11 FINAL REPORT VALIDATION")
print("=" * 65)
print("Report:", report_file.name)
print("Report exists:", report_file.exists())
print("Report size:", report_file.stat().st_size, "bytes")
print("Required CSV files:", len(required_output_names))
print("Missing CSV files:", missing_outputs)

assert report_file.exists()
assert len(saved_report_text) > 1000
assert not missing_outputs
assert "synthetic" in saved_report_text.lower()
assert "business validation" in saved_report_text.lower()

print("\nPASS: Day 11 report created and validated.")

DAY 11 FINAL REPORT VALIDATION
Report: Day11_Recommendation_Anomaly_Detection_Report.md
Report exists: True
Report size: 6329 bytes
Required CSV files: 9
Missing CSV files: []

PASS: Day 11 report created and validated.
